# Phase 5 — Algorithmic Fairness Audit

**Thesis:** Tri-Modal Depression Risk Detection  
**Author:** Md. Mursalin, Netrokona University

## Why fairness matters in clinical AI

> A depression screening model that is 80% accurate *overall* but systematically
> misses depression in one gender group is **unsafe to deploy**. Aggregate accuracy
> hides group-level harm. This notebook audits the trained tri-modal model against
> the standard group-fairness criteria.

### Fairness criteria (Barocas, Hardt & Narayanan, 2019)

| Criterion | Definition | Clinical meaning |
|-----------|-----------|------------------|
| **Demographic Parity** | P(Ŷ=1\|M) ≈ P(Ŷ=1\|F) | Equal flagging rate across genders |
| **Equal Opportunity** | TPR_M ≈ TPR_F | Equal detection of *truly depressed* people |
| **Equalized Odds** | (TPR,FPR)_M ≈ (TPR,FPR)_F | Equal true- AND false-positive rates |
| **Predictive Parity** | PPV_M ≈ PPV_F | A positive flag means the same thing for both |

A **gap < 0.10** is treated as fair; **< 0.05** as strongly fair.

In [ ]:
!git clone https://github.com/DevMursLab/Thesis.git depression_thesis
%cd depression_thesis
!pip install -q -r requirements.txt

## Prerequisite

Phase 4 must have produced `results/fusion_trimodal_best.pth`.
Run `notebooks/phase4_fusion.ipynb` first (or `train_fusion.py`).

In [ ]:
# Runs the full fairness audit:
#   - per-group TPR / FPR / PPV / F1 / AUC
#   - all 6 fairness-criterion gaps with PASS/FAIL verdict
#   - saves 3 figures + JSON report
!python src/fairness/fairness_analysis.py

## Visualize the generated figures

In [ ]:
from IPython.display import Image, display
for f in ['phase5_group_metrics.png',
          'phase5_fairness_gaps.png',
          'phase5_roc_per_group.png']:
    display(Image(f'results/figures/{f}'))

## Inspect the JSON fairness report

In [ ]:
import json
with open('results/metrics/phase5_fairness.json') as f:
    report = json.load(f)
print(json.dumps(report['gaps'], indent=2))
print('\nVerdict:', report['criteria_passed'], 'criteria within fairness threshold')
for k, v in report['verdict'].items():
    print(f'  {k:28s} {v}')

## Interpreting the result

- **All gaps below threshold** → the fairness loss (`λ·(TPR_M − TPR_F)²`) succeeded
  in equalizing the model across gender groups.
- **A specific gap above threshold** → identifies *which* fairness notion is violated,
  guiding targeted mitigation (e.g. raising λ, reweighting, threshold adjustment per group).

> Note: on the small CPU baseline the model may collapse to a single class — in that case
> all groups are *equally* (trivially) treated. Meaningful fairness numbers require the
> full GPU training run from Phase 4. This notebook provides the **audit framework**;
> the numbers update automatically once a properly trained model is supplied.